# Baselines

Two baselines against which QLoRA adapters are compared:
- **B1 -- Zero-shot:** Base model with artist name in prompt, no fine-tuning
- **B2 -- Few-shot:** Base model with artist name + 3 example lyrics in prompt

In [ ]:
import torch
import pandas as pd
import random

from generation.model import load_base_model
from classifier.model import load_classifier
from classifier.classify import classify
from config import ARTISTS

model, tokenizer = load_base_model()
clf = load_classifier()

artists = ARTISTS
n_samples = 10

In [ ]:
def generate_samples(prompt, n):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    samples = []
    for _ in range(n):
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=512,
                min_new_tokens=200,
                do_sample=True,
                temperature=0.9,
                top_p=0.9,
                repetition_penalty=1.1,
            )
        text = tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True)
        samples.append(text)
    return samples, prompt_len


def classify_samples(samples):
    return pd.DataFrame([classify(clf, text) for text in samples])

## B1 -- Zero-shot

Artist name in prompt, no examples, no fine-tuning.

In [3]:
zero_shot = {}

for artist in artists:
    prompt = f"Write song lyrics in the style of {artist}.\n\n"
    samples, prompt_len = generate_samples(prompt, n_samples)
    df = classify_samples(samples)
    zero_shot[artist] = {"samples": samples, "df": df}

    print(f"\n=== {artist} (zero-shot, prompt: {prompt_len} tokens) ===")
    print(f"Mean attribution:\n{df.mean().round(4)}")
    print(f"Std:\n{df.std().round(4)}")

/home/aliozkaya/uni/467/term_project/src/.venv/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



=== Gojira (zero-shot, prompt: 12 tokens) ===
Mean attribution:
Death        0.0220
Gojira       0.0105
Meshuggah    0.0149
Opeth        0.0036
Tool         0.9490
dtype: float64
Std:
Death        0.0335
Gojira       0.0075
Meshuggah    0.0146
Opeth        0.0027
Tool         0.0556
dtype: float64

=== Tool (zero-shot, prompt: 11 tokens) ===
Mean attribution:
Death        0.0054
Gojira       0.0149
Meshuggah    0.0065
Opeth        0.0021
Tool         0.9711
dtype: float64
Std:
Death        0.0038
Gojira       0.0311
Meshuggah    0.0035
Opeth        0.0005
Tool         0.0343
dtype: float64


## B2 -- Few-shot (3 examples)

Artist name + 3 training lyrics as in-context examples.

In [4]:
train_df = pd.read_csv("./data/train.csv")
n_examples = 3
random.seed(42)

few_shot = {}

for artist in artists:
    artist_lyrics = train_df[train_df["artist"] == artist]["lyrics"].tolist()
    examples = random.sample(artist_lyrics, n_examples)

    prompt = f"Write song lyrics in the style of {artist}.\n\n"
    for i, ex in enumerate(examples, 1):
        prompt += f"Example {i}:\n{ex}\n\n"
    prompt += f"Now write new song lyrics in the style of {artist}:\n\n"

    samples, prompt_len = generate_samples(prompt, n_samples)
    df = classify_samples(samples)
    few_shot[artist] = {"samples": samples, "df": df}

    print(f"\n=== {artist} (few-shot, {n_examples} examples, prompt: {prompt_len} tokens) ===")
    print(f"Mean attribution:\n{df.mean().round(4)}")
    print(f"Std:\n{df.std().round(4)}")

/home/aliozkaya/uni/467/term_project/src/.venv/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



=== Gojira (few-shot, 3 examples, prompt: 566 tokens) ===
Mean attribution:
Death        0.0087
Gojira       0.0830
Meshuggah    0.0076
Opeth        0.0023
Tool         0.8984
dtype: float64
Std:
Death        0.0044
Gojira       0.2345
Meshuggah    0.0040
Opeth        0.0007
Tool         0.2395
dtype: float64

=== Tool (few-shot, 3 examples, prompt: 1364 tokens) ===
Mean attribution:
Death        0.0192
Gojira       0.0129
Meshuggah    0.0176
Opeth        0.0032
Tool         0.9471
dtype: float64
Std:
Death        0.0246
Gojira       0.0084
Meshuggah    0.0147
Opeth        0.0023
Tool         0.0441
dtype: float64


## Summary

In [5]:
print("Target-artist mean attribution (higher = better):\n")
print(f"{'Artist':<12} {'Zero-shot':>10} {'Few-shot':>10}")
print("-" * 34)
for artist in artists:
    zs = zero_shot[artist]["df"][artist].mean()
    fs = few_shot[artist]["df"][artist].mean()
    print(f"{artist:<12} {zs:>10.4f} {fs:>10.4f}")

Target-artist mean attribution (higher = better):

Artist        Zero-shot   Few-shot
----------------------------------
Gojira           0.0105     0.0830
Tool             0.9711     0.9471


In [ ]:
import json
from config import RESULTS_DIR

# Persist baselines so 06_evaluation reads them instead of hardcoding numbers.
RESULTS_DIR.mkdir(exist_ok=True)
baselines = {}
for artist in artists:
    zs = zero_shot[artist]["df"][artist]
    fs = few_shot[artist]["df"][artist]
    baselines[artist] = {
        "zero_shot": {"mean": float(zs.mean()), "std": float(zs.std())},
        "few_shot":  {"mean": float(fs.mean()), "std": float(fs.std())},
    }

with open(RESULTS_DIR / "baselines.json", "w") as f:
    json.dump(baselines, f, indent=2)
print(f"wrote {RESULTS_DIR / 'baselines.json'}")